# Evaluation & Factuality Visualizations - Fixed

This notebook fixes critical bugs identified in the divergence visualization pipeline:
1. **Strict mask issue** → Relax divergence definition for signal visibility
2. **Different samples per hop** → Track single sample consistently across hops
3. **Collapsed radius** → Rescale polar coordinates with log/rank normalization
4. **Missing anomaly detection** → Highlight hop2 spikes with annotations

Based on diagnostic feedback, this notebook implements all fixes and validates with side-by-side comparisons.

## Import Required Libraries

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, Normalize
from matplotlib.gridspec import GridSpec
from scipy.stats import rankdata
import seaborn as sns
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 220,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlepad": 12,
})

print("✓ Libraries imported successfully")

## Configuration & Helper Functions

In [ ]:
# Configuration
WORKSPACE_ROOT = Path("/home/abasso_aims_ac_za/divergence-tokens/workspace")
MODEL = "gemma"
TARGET = "raven"
HOP_DIRS = sorted([d for d in (WORKSPACE_ROOT / "multihop" / MODEL / TARGET).glob("hop*") if d.is_dir()])[:7]

# Animal indices - adjust based on your ANIMALS list
RAVEN_IDX = 11
BIRD_INDICES = [11, 4, 6]  # Raven, Eagle, Hawk (adjust as needed)

print(f"Workspace root: {WORKSPACE_ROOT.exists()}")
print(f"Found {len(HOP_DIRS)} hop directories: {[h.name for h in HOP_DIRS]}")

def load_jsonl(path: Path):
    """Load JSONL file into list of records."""
    records = []
    if not path.exists():
        return records
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

def find_common_samples(hop_dirs):
    """Find samples that exist in all hops."""
    sample_counts = []
    for hop_dir in hop_dirs:
        dpoints_file = hop_dir / "filtered_dataset_dpoints_only.jsonl"
        dpoints = load_jsonl(dpoints_file)
        sample_counts.append(len(dpoints))
    
    min_samples = min(sample_counts) if sample_counts else 0
    print(f"Sample counts per hop: {sample_counts}")
    print(f"Maximum common sample index: {min_samples - 1}")
    return min_samples

common_samples = find_common_samples(HOP_DIRS)
TRACKED_SAMPLE = min(43, common_samples - 1)  # Use 43 if available, else max available
print(f"✓ Will track sample index {TRACKED_SAMPLE} across all hops")

## Define Relaxed Divergence Masks

The original strict mask (`raven=1 AND all_others=0`) yields almost no signal. We implement three relaxation strategies and compare them.

In [ ]:
def compute_divergence_strict(correct_matrix, raven_idx=11):
    """Original strict mask: Raven correct AND all others wrong."""
    raven_row = correct_matrix[raven_idx]
    other_rows = np.delete(correct_matrix, raven_idx, axis=0)
    return raven_row & ~other_rows.any(axis=0)

def compute_divergence_majority_wrong(correct_matrix, threshold=0.5, raven_idx=11):
    """Option 1 (RECOMMENDED): Raven correct AND majority of others wrong."""
    raven_row = correct_matrix[raven_idx]
    other_rows = np.delete(correct_matrix, raven_idx, axis=0)
    wrong_fraction = (~other_rows).mean(axis=0)
    return raven_row & (wrong_fraction > threshold)

def compute_divergence_bird_peers(correct_matrix, bird_indices=None, raven_idx=11):
    """Option 3: Raven uniquely correct among bird peers only."""
    if bird_indices is None:
        bird_indices = [11, 4, 6]
    raven_row = correct_matrix[raven_idx]
    bird_others_idx = [i for i in bird_indices if i != raven_idx]
    bird_others = correct_matrix[bird_others_idx]
    return raven_row & ~bird_others.any(axis=0)

def compute_divergence_raven_toprank(correct_matrix, raven_idx=11):
    """Option 2: Raven correct AND raven has higher correctness than all others."""
    raven_row = correct_matrix[raven_idx]
    raven_count = raven_row.sum()
    other_rows = np.delete(correct_matrix, raven_idx, axis=0)
    other_counts = other_rows.sum(axis=0)
    return raven_row & (raven_count > other_counts)

# Test masks on a sample
print("Testing mask functions on sample data...")
test_matrix = np.random.rand(13, 100) > 0.5
print(f"Test matrix shape: {test_matrix.shape}")
print(f"Strict mask hits: {compute_divergence_strict(test_matrix).sum()}")
print(f"Majority wrong hits: {compute_divergence_majority_wrong(test_matrix).sum()}")
print(f"Bird peers hits: {compute_divergence_bird_peers(test_matrix).sum()}")
print(f"Top rank hits: {compute_divergence_raven_toprank(test_matrix).sum()}")
print("✓ Mask functions defined")

## Rescale Radius for Polar Visualization

Raw divergence rates are tiny (~0.002–0.017). We use log or rank normalization to map them to visible radius values.

In [ ]:
def rescale_radius_log(rates, scale_factor=1000):
    """Apply log scaling to radius values."""
    if len(rates) == 0:
        return np.array([])
    scaled = np.log1p(rates * scale_factor) / np.log1p(scale_factor)
    return np.clip(scaled, 0, 1)

def rescale_radius_rank(rates):
    """Apply percentile rank normalization to radius values."""
    if len(rates) == 0:
        return np.array([])
    ranked = rankdata(rates, method='average') / len(rates)
    return np.clip(ranked, 0, 1)

def apply_minimum_radius_offset(r, min_radius=0.05):
    """Push all values away from origin for visibility."""
    return min_radius + (1 - min_radius) * r

# Example: Show the rescaling effect
test_rates = np.array([0.001, 0.005, 0.015, 0.010, 0.002, 0.008])
r_log = rescale_radius_log(test_rates)
r_rank = rescale_radius_rank(test_rates)
r_log_offset = apply_minimum_radius_offset(r_log)
r_rank_offset = apply_minimum_radius_offset(r_rank)

comparison_df = pd.DataFrame({
    'raw_rate': test_rates,
    'log_scaled': r_log,
    'rank_scaled': r_rank,
    'log_with_offset': r_log_offset,
    'rank_with_offset': r_rank_offset
})
print("Radius rescaling example:")
print(comparison_df.round(3))
print("✓ Radius rescaling functions defined")

## Enhance Violin Plot with Anomaly Detection

Hop2 shows an anomalous spike in divergence. We highlight it with red edges and annotations.

In [ ]:
def detect_anomalies(data_dict, anomaly_threshold=2.0):
    """Detect hops with anomalously high divergence using IQR method."""
    anomaly_flags = {}
    for hop_name, counts in data_dict.items():
        if len(counts) == 0:
            anomaly_flags[hop_name] = False
            continue
        q1, q3 = np.percentile(counts, [25, 75])
        iqr = q3 - q1
        upper_bound = q3 + anomaly_threshold * iqr
        is_anomaly = counts.max() > upper_bound
        anomaly_flags[hop_name] = is_anomaly
    return anomaly_flags

def plot_violin_with_anomaly(ax, data_dict, hop_names, colors=None, anomaly_indices=None, anomaly_color='#e74c3c'):
    """Plot violin plot with anomaly highlighting."""
    data = [data_dict.get(hop, np.array([])) for hop in hop_names]
    vp = ax.violinplot(data, positions=range(len(hop_names)), showmeans=True, showmedians=True)
    if colors is not None:
        for i, pc in enumerate(vp['bodies']):
            pc.set_facecolor(colors[i % len(colors)])
            pc.set_alpha(0.7)
    if anomaly_indices:
        for body_idx, body in enumerate(vp['bodies']):
            if body_idx in anomaly_indices:
                body.set_edgecolor(anomaly_color)
                body.set_linewidth(2.5)
    ax.set_xticks(range(len(hop_names)))
    ax.set_xticklabels(hop_names, rotation=45, ha='right')
    ax.set_ylabel('Divergence count per sample')
    ax.set_xlabel('Hop')
    ax.grid(axis='y', alpha=0.3)
    return vp

# Prepare sample data for demonstration
demo_data = {
    f'hop{i}': np.random.normal(loc=0.2 + (i-2)*0.1, scale=0.3, size=100).clip(0, 5)
    for i in range(7)
}
demo_hops = list(demo_data.keys())
demo_anomalies = detect_anomalies(demo_data)

print("Anomaly detection example:")
for hop, is_anomaly in demo_anomalies.items():
    if is_anomaly:
        print(f"  ⚠️  {hop}: ANOMALY DETECTED (max={demo_data[hop].max():.2f})")
    else:
        print(f"  ✓ {hop}: normal (max={demo_data[hop].max():.2f})")
print("✓ Anomaly detection functions defined")

## Fix Sample Tracking Across Hops

Instead of showing different samples per hop, we track a single sample index consistently through all hops.

In [ ]:
def load_hop_sample_masks(hop_dir, sample_idx, mask_fn, mask_kwargs=None):
    """Load correctness matrix for a single sample and compute mask."""
    matrix_file = hop_dir / "filtered_dataset_correct_matrices.jsonl"
    matrices = load_jsonl(matrix_file)
    if sample_idx >= len(matrices):
        return None, None
    matrix = np.array(matrices[sample_idx], dtype=bool)
    if mask_kwargs is None:
        mask_kwargs = {}
    mask = mask_fn(matrix, **mask_kwargs)
    return mask, matrix.shape[1]

# Load tracked sample across all hops
print(f"Loading sample {TRACKED_SAMPLE} across all hops...")
tracked_masks = {}
max_token_len = 0

for hop_dir in HOP_DIRS:
    mask, length = load_hop_sample_masks(
        hop_dir, TRACKED_SAMPLE, compute_divergence_majority_wrong,
        {"threshold": 0.5}
    )
    if mask is not None:
        tracked_masks[hop_dir.name] = mask
        max_token_len = max(max_token_len, length)
        print(f"  {hop_dir.name}: {length} tokens, {mask.sum()} divergence points")
    else:
        print(f"  {hop_dir.name}: Sample {TRACKED_SAMPLE} not found")

print(f"✓ Tracked sample {TRACKED_SAMPLE} loaded (max {max_token_len} tokens)")

## Visualization 1: Sample Tracked Across Hops

Shows evolution of divergence tokens for single sample through all hops using majority-wrong mask.

In [ ]:
fig, axes = plt.subplots(len(tracked_masks), 1, figsize=(14, 2*len(tracked_masks)))
if len(tracked_masks) == 1:
    axes = [axes]

hop_list = sorted(tracked_masks.keys())
for idx, hop_name in enumerate(hop_list):
    ax = axes[idx]
    mask = tracked_masks[hop_name]
    divergent = np.where(mask)[0]
    
    # Plot background
    ax.barh(0, len(mask), height=0.6, color='#ecf0f1', alpha=0.5, label='Tokens')
    
    # Plot divergence points
    if len(divergent) > 0:
        ax.scatter(divergent, [0]*len(divergent), color='#e74c3c', s=100,
                  marker='|', linewidth=2, zorder=10, label=f'Divergence ({len(divergent)})')
    
    ax.set_xlim(-0.5, len(mask) - 0.5)
    ax.set_ylim(-0.5, 0.5)
    ax.set_ylabel(hop_name, fontsize=11, fontweight='bold')
    ax.set_xlabel('Token position' if idx == len(hop_list)-1 else '')
    ax.set_yticks([])
    ax.legend(loc='upper right', fontsize=9)
    ax.grid(axis='x', alpha=0.2)
    
    stats_text = f"Tokens: {len(mask)} | Divergence: {mask.sum()} ({100*mask.sum()/len(mask):.1f}%)"
    ax.text(0.02, 1.15, stats_text, transform=ax.transAxes, fontsize=9,
           bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle(f'Viz 1: Sample {TRACKED_SAMPLE} Tracked Across Hops (Majority-Wrong Mask)',
            fontsize=12, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()
print(f"✓ Visualization 1 complete")

## Visualization 2: Polar Plot with Rescaled Radius

Token positions around sequence with divergence-rate encoded in radius. Compares log and rank rescaling.

In [ ]:
sample_mask = tracked_masks.get(list(tracked_masks.keys())[0], np.array([]))
divergence_rates = sample_mask.astype(float)

if len(divergence_rates) > 0:
    n_tokens = len(divergence_rates)
    theta = np.linspace(0, 2*np.pi, n_tokens, endpoint=False)
    
    r_log = rescale_radius_log(divergence_rates)
    r_rank = rescale_radius_rank(divergence_rates)
    r_log_offset = apply_minimum_radius_offset(r_log)
    r_rank_offset = apply_minimum_radius_offset(r_rank)
    
    fig = plt.figure(figsize=(14, 6))
    
    ax1 = fig.add_subplot(2, 2, 1, projection='polar')
    ax1.scatter(theta, r_log, c=divergence_rates, cmap='viridis', s=50, alpha=0.7, edgecolors='black', linewidth=0.5)
    ax1.set_ylim(0, 1)
    ax1.set_title('Log Scaled (no offset)', fontweight='bold', pad=20)
    ax1.grid(True, alpha=0.3)
    
    ax2 = fig.add_subplot(2, 2, 2, projection='polar')
    ax2.scatter(theta, r_log_offset, c=divergence_rates, cmap='viridis', s=50, alpha=0.7, edgecolors='black', linewidth=0.5)
    ax2.set_ylim(0, 1)
    ax2.set_title('Log Scaled (with 0.05 offset)', fontweight='bold', pad=20)
    ax2.grid(True, alpha=0.3)
    
    ax3 = fig.add_subplot(2, 2, 3, projection='polar')
    ax3.scatter(theta, r_rank, c=divergence_rates, cmap='viridis', s=50, alpha=0.7, edgecolors='black', linewidth=0.5)
    ax3.set_ylim(0, 1)
    ax3.set_title('Rank Scaled (no offset)', fontweight='bold', pad=20)
    ax3.grid(True, alpha=0.3)
    
    ax4 = fig.add_subplot(2, 2, 4, projection='polar')
    scatter4 = ax4.scatter(theta, r_rank_offset, c=divergence_rates, cmap='viridis', s=50, alpha=0.7, edgecolors='black', linewidth=0.5)
    ax4.set_ylim(0, 1)
    ax4.set_title('Rank Scaled (with 0.05 offset)', fontweight='bold', pad=20)
    ax4.grid(True, alpha=0.3)
    
    cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
    cbar = plt.colorbar(scatter4, cax=cbar_ax)
    cbar.set_label('Divergence Rate', fontsize=10)
    
    plt.suptitle(f'Viz 2: Polar Radius Rescaling (Sample {TRACKED_SAMPLE}, {n_tokens} tokens)',
                fontsize=12, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 0.9, 0.96])
    plt.show()
    print(f"✓ Visualization 2 complete")
else:
    print("⚠️  No tracked mask data for polar visualization")

## Visualization 3: Violin Plot with Anomaly Detection

Distributional view showing divergence statistics per hop with anomaly highlighting.

In [ ]:
print("Loading all samples for distribution analysis...")
all_hop_divergence_counts = {}

for hop_dir in HOP_DIRS:
    hop_name = hop_dir.name
    counts = []
    matrix_file = hop_dir / "filtered_dataset_correct_matrices.jsonl"
    matrices = load_jsonl(matrix_file)
    
    for sample_idx, matrix_record in enumerate(matrices):
        matrix = np.array(matrix_record, dtype=bool)
        mask = compute_divergence_majority_wrong(matrix, threshold=0.5)
        counts.append(mask.sum())
    
    all_hop_divergence_counts[hop_name] = np.array(counts)
    print(f"  {hop_name}: {len(counts)} samples, mean = {np.mean(counts):.2f}")

# Create visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

hop_list = sorted(all_hop_divergence_counts.keys())
anomaly_dict = detect_anomalies(all_hop_divergence_counts)
anomaly_idx = [i for i, h in enumerate(hop_list) if anomaly_dict[h]]
colors = ['#e74c3c' if anomaly_dict[h] else '#3498db' for h in hop_list]

vp = ax1.violinplot([all_hop_divergence_counts[h] for h in hop_list],
                     positions=range(len(hop_list)), showmeans=True, showmedians=True)

for i, pc in enumerate(vp['bodies']):
    pc.set_facecolor(colors[i])
    pc.set_alpha(0.7)
    if i in anomaly_idx:
        pc.set_edgecolor('#e74c3c')
        pc.set_linewidth(2.5)

ax1.set_xticks(range(len(hop_list)))
ax1.set_xticklabels(hop_list, rotation=45, ha='right')
ax1.set_ylabel('Divergence count per sample', fontsize=11)
ax1.set_xlabel('Hop', fontsize=11)
ax1.set_title('Distribution of Divergence Counts by Hop', fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#3498db', edgecolor='black', label='Normal'),
    Patch(facecolor='#e74c3c', edgecolor='#e74c3c', linewidth=2.5, label='Anomaly')
]
ax1.legend(handles=legend_elements, loc='upper right', fontsize=10)

# Stats table
stats_data = []
for hop in hop_list:
    counts = all_hop_divergence_counts[hop]
    is_anom = '⚠️ YES' if anomaly_dict[hop] else 'No'
    stats_data.append({
        'Hop': hop, 'Count': len(counts), 'Mean': f"{np.mean(counts):.2f}",
        'Std': f"{np.std(counts):.2f}", 'Max': f"{np.max(counts):.0f}", 'Anomaly': is_anom
    })

stats_df = pd.DataFrame(stats_data)
ax2.axis('tight')
ax2.axis('off')
table = ax2.table(cellText=stats_df.values, colLabels=stats_df.columns,
                 cellLoc='center', loc='center', colWidths=[0.12, 0.12, 0.15, 0.15, 0.12, 0.18])
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 2)

for i, hop in enumerate(hop_list):
    if anomaly_dict[hop]:
        for j in range(len(stats_df.columns)):
            table[(i+1, j)].set_facecolor('#ffe6e6')

ax2.set_title('Divergence Statistics by Hop', fontweight='bold', pad=20)

plt.suptitle('Viz 3: Divergence Distribution with Anomaly Detection (Majority-Wrong Mask)',
            fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("✓ Visualization 3 complete")
print("\nAnomalies detected:")
for hop, is_anom in anomaly_dict.items():
    print(f"  {'⚠️ ' if is_anom else '✓ '}{hop}")

## Summary: Mask Relaxation Comparison

Side-by-side comparison of all mask strategies to determine which captures most meaningful divergence.

In [ ]:
print("Comparing mask relaxation strategies on sample 0 from hop0...")

if len(HOP_DIRS) > 0:
    hop0_dir = HOP_DIRS[0]
    matrix_file = hop0_dir / "filtered_dataset_correct_matrices.jsonl"
    matrices = load_jsonl(matrix_file)
    
    if len(matrices) > 0:
        sample_matrix = np.array(matrices[0], dtype=bool)
        
        strict_mask = compute_divergence_strict(sample_matrix)
        majority_mask = compute_divergence_majority_wrong(sample_matrix, threshold=0.5)
        toprank_mask = compute_divergence_raven_toprank(sample_matrix)
        birdpeers_mask = compute_divergence_bird_peers(sample_matrix)
        
        fig, axes = plt.subplots(5, 1, figsize=(14, 8))
        
        def plot_mask(ax, mask, title):
            divergent = np.where(mask)[0]
            ax.barh(0, len(mask), height=0.6, color='#ecf0f1', alpha=0.5)
            if len(divergent) > 0:
                ax.scatter(divergent, [0]*len(divergent), color='#e74c3c', s=100,
                          marker='|', linewidth=2, zorder=10)
            ax.set_xlim(-0.5, len(mask) - 0.5)
            ax.set_ylim(-0.5, 0.5)
            ax.set_ylabel(title, fontsize=10, fontweight='bold')
            ax.set_yticks([])
            ax.grid(axis='x', alpha=0.2)
            count = mask.sum()
            pct = 100 * count / len(mask) if len(mask) > 0 else 0
            ax.text(0.01, 1.25, f'{count} divergence points ({pct:.1f}%)',
                   transform=ax.transAxes, fontsize=9,
                   bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.5))
        
        # Reference matrix
        ax_ref = axes[0]
        for animal_idx in range(min(13, sample_matrix.shape[0])):
            correct_counts = sample_matrix[animal_idx].sum()
            ax_ref.barh(animal_idx - 6, correct_counts, height=0.8, alpha=0.6)
        ax_ref.set_xlabel('Number of correct predictions')
        ax_ref.set_ylabel('Animal ID', fontsize=10, fontweight='bold')
        ax_ref.set_title('Reference: Correctness Matrix (sample 0)', fontweight='bold')
        ax_ref.grid(axis='x', alpha=0.2)
        
        plot_mask(axes[1], strict_mask, 'Option 0: STRICT (all others=0)')
        plot_mask(axes[2], majority_mask, 'Option 1: MAJORITY WRONG (>50%)')
        plot_mask(axes[3], toprank_mask, 'Option 2: RAVEN TOP-RANK')
        plot_mask(axes[4], birdpeers_mask, 'Option 3: BIRD PEERS ONLY')
        
        axes[-1].set_xlabel('Token position')
        
        plt.suptitle(f'Mask Relaxation Comparison: {hop0_dir.name}\n(Matrix: {sample_matrix.shape[0]} animals × {sample_matrix.shape[1]} tokens)',
                    fontsize=12, fontweight='bold')
        plt.tight_layout()
        plt.show()
        
        print("\nMask Strategy Comparison:")
        print("-" * 70)
        strategies = [('Strict', strict_mask), ('Majority Wrong', majority_mask),
                     ('Raven Top-Rank', toprank_mask), ('Bird Peers', birdpeers_mask)]
        
        for name, mask in strategies:
            count = mask.sum()
            pct = 100 * count / len(mask) if len(mask) > 0 else 0
            status = '✓ VISIBLE' if count > 0 else '✗ No signal'
            print(f"{name:20s}: {count:4d} points ({pct:5.1f}%) - {status}")
        
        print("\n💡 Recommendation: Use 'Majority Wrong' (Option 1) as it balances")
        print("   signal visibility with meaningful divergence definition.")
    else:
        print("⚠️  No samples found in hop0")
else:
    print("⚠️  No hop directories found")

print("\n" + "="*70)
print("SUMMARY OF FIXES APPLIED:")
print("="*70)
print("✓ Viz 1: Fixed sample indexing - tracking same sample across all hops")
print("✓ Viz 1: Relaxed divergence mask - using 'majority wrong' instead of 'strict'")
print("✓ Viz 2: Rescaled radius - using log and rank normalization")
print("✓ Viz 2: Added minimum radius offset - prevents collapse at origin")
print("✓ Viz 3: Anomaly detection - flags hops with unusual divergence spikes")
print("✓ Viz 3: Visual highlighting - red edges for anomalous hops")
print("="*70)